In [1]:
import json
import queue
import random
import time
from pathlib import Path

import carla

OUTPUT_ROOT = Path("output/exp02")
RGB_DIR = OUTPUT_ROOT / "rgb"
SEG_DIR = OUTPUT_ROOT / "seg"

TARGET_PAIRS = 30
AUTOPILOT_WARMUP_TICKS = 20
CAPTURE_INTERVAL_TICKS = 8
SYNC_DELTA = 0.05
TRAFFIC_MANAGER_PORT = 8000
IGNORE_LIGHTS_PERCENTAGE = 100.0
IMAGE_SIZE_X = "800"
IMAGE_SIZE_Y = "600"
CAMERA_FOV = "90"

In [2]:
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)
world = client.get_world()
bp_lib = world.get_blueprint_library()

RGB_DIR.mkdir(parents=True, exist_ok=True)
SEG_DIR.mkdir(parents=True, exist_ok=True)

actor_list = []
original_settings = world.get_settings()
rgb_queue = queue.Queue(maxsize=1)
seg_queue = queue.Queue(maxsize=1)
traffic_manager = None
ego_vehicle = None

In [3]:
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = SYNC_DELTA
world.apply_settings(settings)

vehicle_bp = bp_lib.find("vehicle.tesla.model3")
vehicle_bp.set_attribute("role_name", "hero")

spawn_points = world.get_map().get_spawn_points()
random.shuffle(spawn_points)

ego_vehicle = None
for spawn_point in spawn_points:
    ego_vehicle = world.try_spawn_actor(vehicle_bp, spawn_point)
    if ego_vehicle is not None:
        break

In [4]:
rgb_bp = bp_lib.find("sensor.camera.rgb")
seg_bp = bp_lib.find("sensor.camera.semantic_segmentation")

for bp in (rgb_bp, seg_bp):
    bp.set_attribute("image_size_x", IMAGE_SIZE_X)
    bp.set_attribute("image_size_y", IMAGE_SIZE_Y)
    bp.set_attribute("fov", CAMERA_FOV)
    bp.set_attribute("sensor_tick", "0.0")

CAMERA_TRANSFORM = carla.Transform(
    carla.Location(x=1.5, y=0.0, z=2.4),
    carla.Rotation(pitch=0.0, yaw=0.0, roll=0.0),
)

In [5]:
rgb_camera = world.spawn_actor(
    rgb_bp,
    CAMERA_TRANSFORM,
    attach_to=ego_vehicle,
    attachment_type=carla.AttachmentType.Rigid,
)
seg_camera = world.spawn_actor(
    seg_bp,
    CAMERA_TRANSFORM,
    attach_to=ego_vehicle,
    attachment_type=carla.AttachmentType.Rigid,
)

actor_list.extend([rgb_camera, seg_camera])

In [6]:
def push_latest(channel_queue, image):
    while not channel_queue.empty():
        try:
            channel_queue.get_nowait()
        except queue.Empty:
            break
    try:
        channel_queue.put_nowait(image)
    except queue.Full:
        pass

rgb_camera.listen(lambda image: push_latest(rgb_queue, image))
seg_camera.listen(lambda image: push_latest(seg_queue, image))

traffic_manager = client.get_trafficmanager(TRAFFIC_MANAGER_PORT)
traffic_manager.set_synchronous_mode(True)
ego_vehicle.set_autopilot(True, traffic_manager.get_port())
traffic_manager.ignore_lights_percentage(ego_vehicle, IGNORE_LIGHTS_PERCENTAGE)

In [7]:
def wait_for_image(channel_queue, target_frame, timeout=2.0):
    deadline = time.time() + timeout
    while time.time() < deadline:
        remaining = max(0.0, deadline - time.time())
        image = channel_queue.get(timeout=remaining)
        if image.frame >= target_frame:
            return image
    raise TimeoutError("等待目标图像超时")

def wait_for_aligned_pair(target_frame, timeout=3.0):
    rgb_image = wait_for_image(rgb_queue, target_frame, timeout=timeout)
    seg_image = wait_for_image(seg_queue, target_frame, timeout=timeout)

    while rgb_image.frame != seg_image.frame:
        if rgb_image.frame < seg_image.frame:
            rgb_image = wait_for_image(rgb_queue, seg_image.frame, timeout=timeout)
        else:
            seg_image = wait_for_image(seg_queue, rgb_image.frame, timeout=timeout)

    return rgb_image, seg_image

In [8]:
for warmup_index in range(AUTOPILOT_WARMUP_TICKS):
    frame_id = world.tick()
    rgb_image, seg_image = wait_for_aligned_pair(frame_id)
    if (warmup_index + 1) % 5 == 0:
        print(
            f"warmup {warmup_index + 1}/{AUTOPILOT_WARMUP_TICKS}: "
            f"rgb={rgb_image.frame}, seg={seg_image.frame}"
        )

warmup 5/20: rgb=383744, seg=383744
warmup 10/20: rgb=383749, seg=383749
warmup 15/20: rgb=383754, seg=383754
warmup 20/20: rgb=383759, seg=383759


In [9]:
captured_frames = []
drive_ticks = 0

while len(captured_frames) < TARGET_PAIRS:
    frame_id = world.tick()
    drive_ticks += 1
    rgb_image, seg_image = wait_for_aligned_pair(frame_id)

    if drive_ticks % CAPTURE_INTERVAL_TICKS != 0:
        continue

    frame_name = f"{rgb_image.frame:06d}.png"
    rgb_path = RGB_DIR / frame_name
    seg_path = SEG_DIR / frame_name

    rgb_image.save_to_disk(str(rgb_path))
    seg_image.save_to_disk(str(seg_path))
    captured_frames.append(rgb_image.frame)

In [10]:
rgb_frames = {path.stem for path in RGB_DIR.glob("*.png")}
seg_frames = {path.stem for path in SEG_DIR.glob("*.png")}
matched_frames = sorted(rgb_frames & seg_frames)

params = {
    "image_size_x": int(IMAGE_SIZE_X),
    "image_size_y": int(IMAGE_SIZE_Y),
    "fov": int(CAMERA_FOV),
    "fixed_delta_seconds": SYNC_DELTA,
    "ignore_lights_percentage": IGNORE_LIGHTS_PERCENTAGE,
    "capture_interval_ticks": CAPTURE_INTERVAL_TICKS,
    "capture_interval_seconds": CAPTURE_INTERVAL_TICKS * SYNC_DELTA,
}

report = {
    "rgb_count": len(rgb_frames),
    "seg_count": len(seg_frames),
    "matched_count": len(matched_frames),
    "missing_in_seg": sorted(rgb_frames - seg_frames),
    "missing_in_rgb": sorted(seg_frames - rgb_frames),
}

In [11]:
def cleanup():
    if traffic_manager is not None and ego_vehicle is not None:
        ego_vehicle.set_autopilot(False, traffic_manager.get_port())
        traffic_manager.set_synchronous_mode(False)

    for actor in reversed(actor_list):
        if isinstance(actor, carla.Sensor):
            actor.stop()
        safe_destroy(actor)

    actor_list.clear()
    world.apply_settings(original_settings)

In [ ]:
image_queue = queue.Queue(maxsize=1)
camera.listen(push_latest_image)
print("相机已切换到队列模式")

In [ ]:
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

latest_image = None
for _ in range(5):
    world.tick()
    latest_image = image_queue.get()

image = latest_image
print("当前图像帧号：", image.frame)

In [ ]:
os.makedirs("output/ch03", exist_ok=True)
filename = f"output/ch03/frame_{image.frame:06d}.png"
image.save_to_disk(filename)
print("已保存：", filename)

In [ ]:
weather_presets = [
    ("ClearNoon", carla.WeatherParameters.ClearNoon),
    ("CloudySunset", carla.WeatherParameters.CloudySunset),
    ("WetCloudyNoon", carla.WeatherParameters.WetCloudyNoon),
    ("HardRainNoon", carla.WeatherParameters.HardRainNoon),
]

In [ ]:
def capture_weather_frame(label, weather):
    world.set_weather(weather)

    latest_image = None
    for _ in range(6):
        world.tick()
        latest_image = image_queue.get()

    path = f"output/ch03/{label}_{latest_image.frame:06d}.png"
    latest_image.save_to_disk(path)
    print(f"{label} 已保存：{path}")

In [ ]:
for label, weather in weather_presets:
    capture_weather_frame(label, weather)

In [ ]:
traffic_manager = client.get_trafficmanager(8000)
traffic_manager.set_synchronous_mode(True)
ego_vehicle.set_autopilot(True, traffic_manager.get_port())

for step in range(20):
    world.tick()
    image = image_queue.get()
    if step % 5 == 0:
        path = f"output/ch03/autopilot_{image.frame:06d}.png"
        image.save_to_disk(path)
        print("自动驾驶采图：", path)

ego_vehicle.set_autopilot(False, traffic_manager.get_port())
traffic_manager.set_synchronous_mode(False)
traffic_manager = None

In [ ]:
cleanup()